Networks are trained with MMD summary space regularization
D=20
N=100
M1: mu ~ N(0, I), y|mu ~ N(mu, I)
M2: mu ~ N(3, I), y|mu ~ N(mu, I)
M3: mu ~ N(0, I), y|mu ~ N(mu, 3I)
M4: mu ~ N(1.5, I), y|mu ~ N(mu, I)
M5: mu ~ N(5, I), y|mu ~ N(mu, I)
M6: mu ~ N(0, I), y|mu ~ N(mu, 5I)
M7: mu ~ N(0, 3I), y|mu ~ N(mu, I)

In [ ]:
import numpy as np
import pandas as pd
import keras
import bayesflow as bf
from pathlib import Path
import pickle
from benchmark.examples.gaussian.datasets.datasets import GetDatasets 
from benchmark.examples.gaussian.datasets.stan_dataset import  StanDataset
from benchmark.examples.gaussian.datasets.calculation import Calculation
import benchmark.examples.gaussian.direct.calculator as BF

In [2]:
from tqdm.auto import tqdm as original_tqdm
import bayesflow.approximators.helpers.samplers as bf_samplers
import bayesflow.approximators.helpers.conditions as bf_conditions

def quiet_tqdm(*args, **kwargs):
    kwargs["disable"] = True
    return original_tqdm(*args, **kwargs)

bf_samplers.tqdm = quiet_tqdm
bf_conditions.tqdm = quiet_tqdm


In [ ]:
RNG=np.random.default_rng(2025)
num_dims=20
num_obs=100
likelihood_std=1
student_df=5
num_datasets=50

In [ ]:
# generate observation datasets
datasets_m1=GetDatasets(obs_mu_prior_mean=0, obs_mu_prior_std=1,
               num_dims=num_dims, num_obs=num_obs, obs_likelihood_std=1, 
               num_datasets=num_datasets,rng=RNG).get_datasets_normal()

datasets_m2=GetDatasets(obs_mu_prior_mean=3, obs_mu_prior_std=1,
               num_dims=num_dims, num_obs=num_obs, obs_likelihood_std=1, 
               num_datasets=num_datasets,rng=RNG).get_datasets_normal()

datasets_m3=GetDatasets(obs_mu_prior_mean=0, obs_mu_prior_std=1,
               num_dims=num_dims, num_obs=num_obs, obs_likelihood_std=3,
               num_datasets=num_datasets,rng=RNG).get_datasets_normal()

datasets_m4=GetDatasets(obs_mu_prior_mean=1.5, obs_mu_prior_std=1,
               num_dims=num_dims, num_obs=num_obs, obs_likelihood_std=1, 
               num_datasets=num_datasets,rng=RNG).get_datasets_normal()

datasets_m5=GetDatasets(obs_mu_prior_mean=5, obs_mu_prior_std=1,
               num_dims=num_dims, num_obs=num_obs, obs_likelihood_std=1, 
               num_datasets=num_datasets,rng=RNG).get_datasets_normal()

datasets_m6=GetDatasets(obs_mu_prior_mean=0, obs_mu_prior_std=1,
               num_dims=num_dims, num_obs=num_obs, obs_likelihood_std=5, 
               num_datasets=num_datasets,rng=RNG).get_datasets_normal()

datasets_m7=GetDatasets(obs_mu_prior_mean=0, obs_mu_prior_std=3,
               num_dims=num_dims, num_obs=num_obs, obs_likelihood_std=1, 
               num_datasets=num_datasets,rng=RNG).get_datasets_normal()
# save observation datasets
def save_pickle(obj, path: str):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f)

save_pickle(datasets_m1,  "/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/results/datasets/m1_s.pkl")
save_pickle(datasets_m2, "/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/results/datasets/m2_s.pkl")
save_pickle(datasets_m3,   "/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/results/datasets/m3_s.pkl")
save_pickle(datasets_m4,   "/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/results/datasets/m4_s.pkl")
save_pickle(datasets_m5,   "/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/results/datasets/m5_s.pkl")
save_pickle(datasets_m6,   "/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/results/datasets/m6_s.pkl")
save_pickle(datasets_m7,   "/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/results/datasets/m7_s.pkl")
# save datasets for Stan and then run "*.r" for m3 neuro network to get gold standard
# save=StanDataset.save_for_stan(datasets_m1,output_dir="/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/stan/json_m1",df=student_df)
# save=StanDataset.save_for_stan(datasets_m2,output_dir="/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/stan/json_m2",df=student_df)
# save=StanDataset.save_for_stan(datasets_m3,output_dir="/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/stan/json_m3",df=student_df)
# save=StanDataset.save_for_stan(datasets_m4,output_dir="/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/stan/json_m4",df=student_df)


✓ Saved 256 datasets with unique IDs

✓ Saved 256 datasets with unique IDs

✓ Saved 256 datasets with unique IDs

✓ Saved 256 datasets with unique IDs


In [ ]:
# read observation datasets
def load_pickle(path: str):
    with open(path, "rb") as f:
        return pickle.load(f)

datasets_m1  = load_pickle("/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/results/datasets/m1_s.pkl")
datasets_m2 = load_pickle("/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/results/datasets/m2_s.pkl")
datasets_m3   = load_pickle("/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/results/datasets/m3_s.pkl")
datasets_m4   = load_pickle("/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/results/datasets/m4_s.pkl")
datasets_m5   = load_pickle("/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/results/datasets/m5_s.pkl")
datasets_m6   = load_pickle("/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/results/datasets/m6_s.pkl")

In [9]:
print(datasets_m1[0].keys())

dict_keys(['mu', 'x', 'id', 'df', 'gold_post_samples_m3', 'gold_log_marginal_m3'])


In [ ]:
# # load json for m3 gold standard
# datasets_m1=StanDataset.load_datasets_from_json(dir_path="/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/stan/stan_output_m1")
# datasets_m2=StanDataset.load_datasets_from_json(dir_path="/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/stan/stan_output_m2")
# datasets_m3=StanDataset.load_datasets_from_json(dir_path="/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/stan/stan_output_m3")
# datasets_m4=StanDataset.load_datasets_from_json(dir_path="/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/stan/stan_output_m4")

In [ ]:
# load approximators
filepath_m1 = Path("/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/networks") / "m1_s_20d_100n.keras"
filepath_m2 = Path("/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/networks") / "m2_s_20d_100n.keras"
filepath_m3 = Path("/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/networks") / "m3_s_20d_100n.keras"
filepath_d = Path("/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/networks") / "direct_s_20d_100n.keras"
approximator_m1 = keras.saving.load_model(filepath_m1)
approximator_m2 = keras.saving.load_model(filepath_m2)
approximator_m3 = keras.saving.load_model(filepath_m3)
approximator_d = keras.saving.load_model(filepath_d)

In [ ]:
# get analytical and npe estimated posterior samples and logml from m1 trained networks
calculation_m1=Calculation(approximator=approximator_m1,mu_prior_mean=0, mu_prior_std=1,
                        num_dims=num_dims,num_obs=num_obs,
                 likelihood_std=1,num_samples=1000,assumed_model="m1")

datasets_m1=calculation_m1.normal_analytical(obs_data=datasets_m1)
datasets_m1=calculation_m1.npe_estimation(obs_data=datasets_m1)

datasets_m2=calculation_m1.normal_analytical(obs_data=datasets_m2)
datasets_m2=calculation_m1.npe_estimation(obs_data=datasets_m2)

datasets_m3=calculation_m1.normal_analytical(obs_data=datasets_m3)
datasets_m3=calculation_m1.npe_estimation(obs_data=datasets_m3)

datasets_m4=calculation_m1.normal_analytical(obs_data=datasets_m4)
datasets_m4=calculation_m1.npe_estimation(obs_data=datasets_m4)

In [11]:
# get analytical and npe estimated posterior samples and logml from m2 trained networks
calculation_m2=Calculation(approximator=approximator_m2,mu_prior_mean=0.5, mu_prior_std=1,
                        num_dims=num_dims,num_obs=num_obs,
                 likelihood_std=likelihood_std,num_samples=2000,assumed_model="m2")

datasets_m1=calculation_m2.normal_analytical(obs_data=datasets_m1)
datasets_m1=calculation_m2.npe_estimation(obs_data=datasets_m1)
datasets_m2=calculation_m2.normal_analytical(obs_data=datasets_m2)
datasets_m2=calculation_m2.npe_estimation(obs_data=datasets_m2)
datasets_m3=calculation_m2.normal_analytical(obs_data=datasets_m3)
datasets_m3=calculation_m2.npe_estimation(obs_data=datasets_m3)
datasets_m4=calculation_m2.normal_analytical(obs_data=datasets_m4)
datasets_m4=calculation_m2.npe_estimation(obs_data=datasets_m4)

In [12]:
# get npe estimated posterior samples and logml from m3 trained networks
calculation_m3=Calculation(approximator=approximator_m3,mu_prior_mean=0, mu_prior_std=1,
                        num_dims=num_dims,num_obs=num_obs, use_student_t=True,df=student_df,
                 likelihood_std=likelihood_std,num_samples=2000,assumed_model="m3")
datasets_m1=calculation_m3.npe_estimation(obs_data=datasets_m1)
datasets_m2=calculation_m3.npe_estimation(obs_data=datasets_m2)
datasets_m3=calculation_m3.npe_estimation(obs_data=datasets_m3)
datasets_m4=calculation_m3.npe_estimation(obs_data=datasets_m4)

In [13]:
# get npe estimated logml by using gold_posterior smaples
# m1
datasets_m1=calculation_m1.npe_estimation_use_gold_posterior(obs_data=datasets_m1)
datasets_m2=calculation_m1.npe_estimation_use_gold_posterior(obs_data=datasets_m2)
datasets_m3=calculation_m1.npe_estimation_use_gold_posterior(obs_data=datasets_m3)
datasets_m4=calculation_m1.npe_estimation_use_gold_posterior(obs_data=datasets_m4)

#m2
datasets_m1=calculation_m2.npe_estimation_use_gold_posterior(obs_data=datasets_m1)
datasets_m2=calculation_m2.npe_estimation_use_gold_posterior(obs_data=datasets_m2)
datasets_m3=calculation_m2.npe_estimation_use_gold_posterior(obs_data=datasets_m3)
datasets_m4=calculation_m2.npe_estimation_use_gold_posterior(obs_data=datasets_m4)

#m3
datasets_m1=calculation_m3.npe_estimation_use_gold_posterior(obs_data=datasets_m1)
datasets_m2=calculation_m3.npe_estimation_use_gold_posterior(obs_data=datasets_m2)
datasets_m3=calculation_m3.npe_estimation_use_gold_posterior(obs_data=datasets_m3)
datasets_m4=calculation_m3.npe_estimation_use_gold_posterior(obs_data=datasets_m4)

In [14]:
print(datasets_m3[0].keys())

dict_keys(['mu', 'x', 'id', 'df', 'gold_post_samples_m3', 'gold_log_marginal_m3', 'gold_log_marginal_m1', 'gold_post_samples_m1', 'npe_post_samples_m1', 'npe_log_marginal_m1', 'gold_log_marginal_m2', 'gold_post_samples_m2', 'npe_post_samples_m2', 'npe_log_marginal_m2', 'npe_post_samples_m3', 'npe_log_marginal_m3', 'npe_log_marginal_gp_m1', 'npe_log_marginal_gp_m2', 'npe_log_marginal_gp_m3'])


In [15]:
# get posterior model probabilities and log BF from direct MC
datasets_m1=BF.direct_get_probs(obs_data=datasets_m1,approximator=approximator_d)
datasets_m2=BF.direct_get_probs(obs_data=datasets_m2,approximator=approximator_d)
datasets_m3=BF.direct_get_probs(obs_data=datasets_m3,approximator=approximator_d)
datasets_m4=BF.direct_get_probs(obs_data=datasets_m4,approximator=approximator_d)

# get gold and npe posterior model probabilities and log BF from indirect MC
datasets_m1=BF.indirect_get_probs(obs_data=datasets_m1)
datasets_m2=BF.indirect_get_probs(obs_data=datasets_m2)
datasets_m3=BF.indirect_get_probs(obs_data=datasets_m3)
datasets_m4=BF.indirect_get_probs(obs_data=datasets_m4)

In [16]:
print(datasets_m3[0].keys())
print(datasets_m3[0]["p_direct"].round(3))
print(datasets_m3[0]["p_gold"].round(3))
print(datasets_m3[0]["p_npe"].round(3))
print(datasets_m3[0]["logBF_12_direct"])

dict_keys(['mu', 'x', 'id', 'df', 'gold_post_samples_m3', 'gold_log_marginal_m3', 'gold_log_marginal_m1', 'gold_post_samples_m1', 'npe_post_samples_m1', 'npe_log_marginal_m1', 'gold_log_marginal_m2', 'gold_post_samples_m2', 'npe_post_samples_m2', 'npe_log_marginal_m2', 'npe_post_samples_m3', 'npe_log_marginal_m3', 'npe_log_marginal_gp_m1', 'npe_log_marginal_gp_m2', 'npe_log_marginal_gp_m3', 'p_direct', 'logBF_12_direct', 'logBF_13_direct', 'logBF_23_direct', 'logBF_12_gold', 'logBF_13_gold', 'logBF_23_gold', 'logBF_12_npe', 'logBF_13_npe', 'logBF_23_npe', 'p_gold', 'p_npe'])
[0. 0. 1.]
[0. 0. 1.]
[0. 0. 1.]
0.0


In [17]:
# save observation datasets
def save_pickle(obj, path: str):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f)

save_pickle(datasets_m1,  "/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/results/datasets/m1.pkl")
save_pickle(datasets_m2, "/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/results/datasets/m2.pkl")
save_pickle(datasets_m3,   "/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/results/datasets/m3.pkl")
save_pickle(datasets_m4,   "/Users/yimingzang/Documents/thesis/benchmark2/benchmark/examples/gaussian/results/datasets/m4.pkl")


In [18]:
# average posterior model probabilities of observation datasets from m1
mean_p_gold = np.mean(np.stack([d["p_gold"] for d in datasets_m1]), axis=0)
mean_p_npe = np.mean(np.stack([d["p_npe"] for d in datasets_m1]), axis=0)
mean_p_direct = np.mean(np.stack([d["p_direct"] for d in datasets_m1]), axis=0)
print("mean p_gold_m1:", mean_p_gold.round(3))
print("mean p_npe_m1:", mean_p_npe.round(3))
print("mean p_direct_m1:", mean_p_direct.round(3))


mean p_gold_m1: [0.825 0.175 0.   ]
mean p_npe_m1: [0.825 0.175 0.   ]
mean p_direct_m1: [0.827 0.173 0.   ]


In [19]:
# average posterior model probabilities of observation datasets from m2
mean_p_gold = np.mean(np.stack([d["p_gold"] for d in datasets_m2]), axis=0)
mean_p_npe = np.mean(np.stack([d["p_npe"] for d in datasets_m2]), axis=0)
mean_p_direct = np.mean(np.stack([d["p_direct"] for d in datasets_m2]), axis=0)
print("mean p_gold_m2:", mean_p_gold.round(3))
print("mean p_npe_m2:", mean_p_npe.round(3))
print("mean p_direct_m2:", mean_p_direct.round(3))

mean p_gold_m2: [0.193 0.807 0.   ]
mean p_npe_m2: [0.193 0.807 0.   ]
mean p_direct_m2: [0.193 0.807 0.   ]


In [20]:
# average posterior model probabilities of observation datasets from m3
mean_p_gold = np.mean(np.stack([d["p_gold"] for d in datasets_m3]), axis=0)
mean_p_npe = np.mean(np.stack([d["p_npe"] for d in datasets_m3]), axis=0)
mean_p_direct = np.mean(np.stack([d["p_direct"] for d in datasets_m3]), axis=0)
print("mean p_gold_m3:", mean_p_gold.round(3))
print("mean p_npe_m3:", mean_p_npe.round(3))
print("mean p_direct_m3:", mean_p_direct.round(3))

mean p_gold_m3: [0. 0. 1.]
mean p_npe_m3: [0. 0. 1.]
mean p_direct_m3: [0. 0. 1.]


In [21]:
# average posterior model probabilities of observation datasets from m4
mean_p_gold = np.mean(np.stack([d["p_gold"] for d in datasets_m4]), axis=0)
mean_p_npe = np.mean(np.stack([d["p_npe"] for d in datasets_m4]), axis=0)
mean_p_direct = np.mean(np.stack([d["p_direct"] for d in datasets_m4]), axis=0)
print("mean p_gold_m4:", mean_p_gold.round(3))
print("mean p_npe_m4:", mean_p_npe.round(3))
print("mean p_direct_m4:", mean_p_direct.round(3))

mean p_gold_m4: [0. 0. 1.]
mean p_npe_m4: [0. 0. 1.]
mean p_direct_m4: [0.    0.756 0.244]


In [22]:
print(datasets_m1[0].keys())

dict_keys(['mu', 'x', 'id', 'df', 'gold_post_samples_m3', 'gold_log_marginal_m3', 'gold_log_marginal_m1', 'gold_post_samples_m1', 'npe_post_samples_m1', 'npe_log_marginal_m1', 'gold_log_marginal_m2', 'gold_post_samples_m2', 'npe_post_samples_m2', 'npe_log_marginal_m2', 'npe_post_samples_m3', 'npe_log_marginal_m3', 'npe_log_marginal_gp_m1', 'npe_log_marginal_gp_m2', 'npe_log_marginal_gp_m3', 'p_direct', 'logBF_12_direct', 'logBF_13_direct', 'logBF_23_direct', 'logBF_12_gold', 'logBF_13_gold', 'logBF_23_gold', 'logBF_12_npe', 'logBF_13_npe', 'logBF_23_npe', 'p_gold', 'p_npe'])
